# 17 — Data Import & Export

Moving data in and out of SurrealDB, at two levels:

1. **JSON Lines** — one record per line, per table or per filtered `QuerySet`:
   `QuerySet.export_jsonl()` / `iter_jsonl()` / `import_jsonl()`, and the same methods on an SDK
   connection (`export_jsonl()`, `import_jsonl()`, `iter_records()`). Record ids, datetimes, decimals,
   UUIDs, durations and bytes are written in a typed form (`{"$record": "..."}`, `{"$datetime": "..."}`,
   ...) so a file loads back with the same types; `typed=False` writes plain JSON for other tools.
2. **Database dump / restore** — the server's own SurrealQL dump through its `/export` and `/import`
   endpoints: `SurrealDBConnectionManager.dump_database()` / `restore_database()` (HTTP only). A dump keeps
   every value as stored (geometry, nanosecond datetimes, `NULL`), which JSON Lines cannot.

The same operations are available from the command line: `surreal-orm export-jsonl`, `import-jsonl`,
`dump` and `restore`.

> **Requires:** SurrealDB 3.2+ and the CBOR protocol (the default).

Outputs are not saved in this notebook: the expected output of each cell is in a trailing comment.

## Setup

Two named connections on the same server: the working database and an empty copy that the restore section
fills. Both are created by the ORM when it connects.

In [ ]:
import os

from surreal_orm import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
URL = f"http://{HOST}:{PORT}"

SurrealDBConnectionManager.set_connection(
    url=URL,
    user="root",
    password="root",
    namespace="examples",
    database="examples_data_io",
)
SurrealDBConnectionManager.add_connection(
    "copy",
    url=URL,
    user="root",
    password="root",
    namespace="examples",
    database="examples_data_io_copy",
)
print("Connections configured.")
# Connections configured.

### Models and sample data

Authors, posts with a `ForeignKey`, a `Decimal` price and a `datetime`, and two `nb_wrote` graph edges.
The cell starts by removing the tables in both databases, so it can be re-run.

In [ ]:
import contextlib
from datetime import UTC, datetime
from decimal import Decimal

from surreal_orm import BaseSurrealModel, ForeignKey, SurrealConfigDict

TABLES = ("nb_wrote", "nb_post", "nb_author")
for name in ("default", "copy"):
    client = await SurrealDBConnectionManager.get_client(name)
    for table in TABLES:
        with contextlib.suppress(Exception):
            await client.query(f"REMOVE TABLE IF EXISTS {table};")


class NBAuthor(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="nb_author")

    id: str | None = None
    name: str = ""


class NBPost(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="nb_post")

    id: str | None = None
    title: str = ""
    published: bool = False
    price: Decimal = Decimal("0")
    author: ForeignKey("NBAuthor") = None  # type: ignore[valid-type]
    created_at: datetime | None = None


authors = {key: NBAuthor(id=key, name=name) for key, name in [("ada", "Ada"), ("grace", "Grace"), ("linus", "Linus")]}
for author in authors.values():
    await author.save()

posts = [
    NBPost(id="p1", title="Graphs 101", published=True, price=Decimal("9.99"), author="nb_author:ada",
           created_at=datetime(2026, 1, 5, 9, 30, tzinfo=UTC)),
    NBPost(id="p2", title="Draft: indexes", published=False, price=Decimal("0"), author="nb_author:ada",
           created_at=datetime(2026, 1, 6, tzinfo=UTC)),
    NBPost(id="p3", title="Compilers", published=True, price=Decimal("19.50"), author="nb_author:grace",
           created_at=datetime(2026, 2, 1, tzinfo=UTC)),
    NBPost(id="p4", title="Draft: COBOL", published=False, price=Decimal("0"), author="nb_author:grace",
           created_at=datetime(2026, 2, 2, tzinfo=UTC)),
    NBPost(id="p5", title="Kernels", published=True, price=Decimal("25.00"), author="nb_author:linus",
           created_at=datetime(2026, 3, 1, tzinfo=UTC)),
    NBPost(id="p6", title="Git internals", published=True, price=Decimal("12.00"), author="nb_author:linus",
           created_at=datetime(2026, 3, 2, tzinfo=UTC)),
]
for post in posts:
    await post.save()

# Two graph edges, on the drafts: the next sections delete and re-import the published posts only.
await authors["ada"].relate("nb_wrote", posts[1], role="author")
await authors["grace"].relate("nb_wrote", posts[3], role="editor")

print(await NBAuthor.objects().count(), "authors,", await NBPost.objects().count(), "posts")
# 3 authors, 6 posts

---

## 1. `QuerySet.export_jsonl()` — a filtered export

The export honours `filter()`, `Q`, `.variables()` and `limit()`, and reads the table in `id` order, in
batches of `batch_size` rows (keyset on `id`), so only one batch is held in memory. Each line is the raw
database record: column names, record links as `$record`, datetimes as `$datetime`, decimals as
`$decimal`.

In [ ]:
import tempfile
from pathlib import Path

WORKDIR = Path(tempfile.mkdtemp(prefix="sorm_nb17_"))
posts_path = WORKDIR / "published_posts.jsonl"

written = await NBPost.objects().filter(published=True).export_jsonl(posts_path)
print(written, "records written")
print(posts_path.read_text(encoding="utf-8").splitlines()[0])
# 4 records written
# {"author":{"$record":["nb_author","ada"]},"created_at":{"$datetime":"2026-01-05T09:30:00.000000Z"},"id":{"$record":["nb_post","p1"]},"price":{"$decimal":"9.99"},"published":true,"title":"Graphs 101"}

## 2. `iter_jsonl(typed=False)` — plain JSON for other tools

`iter_jsonl()` yields the same lines without writing a file. With `typed=False` the values are plain JSON
(a record id is its `table:id` text, a decimal its exact digits as a string, a datetime its ISO text), ready for `json.loads`, a
dataframe or another system. A plain file does not load back with the same types: keep the default
`typed=True` for files meant for `import_jsonl()`.

In [ ]:
import json

rows = [json.loads(line) async for line in NBPost.objects().filter(published=True).iter_jsonl(typed=False)]
print(len(rows), "rows")
print(rows[0])
# 4 rows
# {'author': 'nb_author:ada', 'created_at': '2026-01-05T09:30:00.000000Z', 'id': 'nb_post:p1', 'price': '9.99', 'published': True, 'title': 'Graphs 101'}

## 3. `QuerySet.import_jsonl()` — loading a file back

Rows are sent in bounded batches, one atomic `INSERT` per batch (`batch_size` rows and at most
`max_batch_bytes` of JSON each). The file as a whole is not atomic: on a failure the batches already
written stay, and `DataImportError.report` says how far the import got.

The rows do not go through the model: no validation, no `save()`, **no signal**. The model's query cache is
cleared. `on_conflict="ignore"` skips ids that already exist; `raw=True` sends each batch under
`OPTION IMPORT` (no schema checks, `VALUE`, `ASSERT` or events; needs a system user).

In [ ]:
deleted = await NBPost.objects().filter(published=True).bulk_delete()
print("deleted:", deleted, "- left:", await NBPost.objects().count())

report = await NBPost.objects().import_jsonl(posts_path)
print(report)
print("posts:", await NBPost.objects().count())

restored = await NBPost.objects().get("p1")
print(type(restored.price).__name__, restored.price, restored.created_at.isoformat(), restored.author)
# deleted: 4 - left: 2
# ImportReport(table='nb_post', lines=4, rows=4, batches=1)
# posts: 6
# Decimal 9.99 2026-01-05T09:30:00+00:00 nb_author:ada

## 4. Graph edges with the SDK client

The SDK connection has the same methods, per table name. An edge keeps its `in` and `out` record links, and
`import_jsonl()` writes it back as an edge (`INSERT RELATION`): it reads the table's type, or, for a table
created by `RELATE`, sees that every row has record `in` and `out` fields (`relation=True` forces it).

In [ ]:
client = await SurrealDBConnectionManager.get_client()
edges_path = WORKDIR / "nb_wrote.jsonl"

print(await client.export_jsonl("nb_wrote", edges_path), "edges exported")
await client.query("DELETE nb_wrote;")
print("after delete:", [p.title for p in await authors["ada"].get_related("nb_wrote", direction="out", model_class=NBPost)])

edge_report = await client.import_jsonl("nb_wrote", edges_path)
print(edge_report)
print("after import:", [p.title for p in await authors["ada"].get_related("nb_wrote", direction="out", model_class=NBPost)])
# 2 edges exported
# after delete: []
# ImportReport(table='nb_wrote', lines=2, rows=2, batches=1)
# after import: ['Draft: indexes']

---

## 5. `dump_database()` — a SurrealQL dump

The dump is the server's own format (what `surreal export` writes): `OPTION IMPORT;`, the definitions, then
the rows. `tables=` takes table names or models; other options leave out users, accesses, params,
functions, analyzers or the rows.

**A full dump contains every user's password hash and the access definitions (JWT keys included)**: pass
`users=False, accesses=False` (CLI: `--no-users --no-accesses`) unless you need them. A new dump file is
created readable by its owner only (`0o600`).

In [ ]:
dump_path = WORKDIR / "blog.surql"

size = await SurrealDBConnectionManager.dump_database(
    dump_path,
    tables=[NBAuthor, NBPost],
    users=False,
    accesses=False,
)
print("bytes written:", size > 0, "- mode:", oct(dump_path.stat().st_mode & 0o777))
for line in dump_path.read_text(encoding="utf-8").splitlines():
    if line.startswith("-- TABLE"):
        print(line)
# bytes written: True - mode: 0o600
# -- TABLE: nb_author
# -- TABLE DATA: nb_author
# -- TABLE: nb_post
# -- TABLE DATA: nb_post

## 6. `restore_database()` — into another database

`using("copy")` routes the restore, and the reads after it, to the copy (`restore_database(path, connection="copy")` does the same for the restore alone). A restore runs under the dump's
`OPTION IMPORT;`: no field processing, no events, **no ORM signal**, and it needs a system user. It is not
transactional: a definition of the same name is replaced, one only in the target is kept, and a row that
already exists fails its `INSERT` (the other statements still run). The whole query cache is cleared
afterwards.

In [ ]:
async with SurrealDBConnectionManager.using("copy"):
    await SurrealDBConnectionManager.restore_database(dump_path)
    copy_counts = (await NBAuthor.objects().count(), await NBPost.objects().count())
source_counts = (await NBAuthor.objects().count(), await NBPost.objects().count())

print("source:", source_counts, "copy:", copy_counts, "equal:", source_counts == copy_counts)
# source: (3, 6) copy: (3, 6) equal: True

---

## Cleanup

In [ ]:
import shutil

for name in ("default", "copy"):
    client = await SurrealDBConnectionManager.get_client(name)
    for table in TABLES:
        with contextlib.suppress(Exception):
            await client.query(f"REMOVE TABLE IF EXISTS {table};")
shutil.rmtree(WORKDIR, ignore_errors=True)
print("Cleaned up.")
# Cleaned up.